In [33]:
from datetime import datetime, timedelta
from pathlib import Path

import ee
import geemap

In [5]:
ee.Authenticate()

ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (failed to map segment from shared object): ignored.
ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/usr/local/lib

Enter verification code:  4/1Aci98E8m7y0A6PBhoDfojbVI3jNQ1ORYKIBlLJ_XK0s39vOkvy2RFVPFxIk



Successfully saved authorization token.



Successfully saved authorization token.


In [6]:
ee.Initialize()

ERROR: ld.so: object '/usr/local/lib/AppProtection/libAppProtection.so' from /etc/ld.so.preload cannot be preloaded (cannot open shared object file): ignored.


Création d'une ee.Geometry() contenant la région Aquitaine (1 seul polygone)


In [7]:
ROI_FILE = Path().resolve().parent / "data" / "geometries" / "aquitaine.geojson"
roi = geemap.geojson_to_ee(str(ROI_FILE))

In [9]:
def list_assets_recursive(parent, indent=0):
    try:
        result = ee.data.listAssets({"parent": parent})
        for asset in result.get("assets", []):
            print(" " * indent + asset["name"].split("/")[-1], f"({asset['type']})")
            if asset["type"] == "FOLDER":
                list_assets_recursive(asset["name"], indent + 2)
    except Exception as e:
        print(f"Error: {e}")

In [15]:
RADD_EUROPE_RESOURCE = "projects/wurnrt-raddeurope/assets/01_NRT/00_Operational/V1_IC"
list_assets_recursive(RADD_EUROPE_RESOURCE)

europe_20250916 (IMAGE)
europe_20260215 (IMAGE)
europe_20260228 (IMAGE)
europe_20260307 (IMAGE)
europe_20260311 (IMAGE)
europe_20260318 (IMAGE)
europe_20260324 (IMAGE)
europe_20260330 (IMAGE)
europe_20260406 (IMAGE)
europe_forestbaseline2019 (IMAGE)


In [16]:
radd_europe_ic = ee.ImageCollection(RADD_EUROPE_RESOURCE)

In [24]:
latest_radd_alert = ee.Image(
    radd_europe_ic.filterMetadata("layer", "contains", "alert")
    .sort("system:time_end", False)
    .first()
)
latest_radd_alert.getInfo()

{'type': 'Image',
 'bands': [{'id': 'Alert',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': -32768,
    'max': 32767},
   'dimensions': [435116, 418403],
   'crs': 'EPSG:3035',
   'crs_transform': [10, 0, 2185410, 0, -10, 5607450]},
  {'id': 'Date',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': -32768,
    'max': 32767},
   'dimensions': [435116, 418403],
   'crs': 'EPSG:3035',
   'crs_transform': [10, 0, 2185410, 0, -10, 5607450]}],
 'version': 1776335915860857,
 'id': 'projects/wurnrt-raddeurope/assets/01_NRT/00_Operational/V1_IC/europe_20260406',
 'properties': {'system:time_start': 1577836800000,
  'geography': 'europe',
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-14.477073804383117, 37.62759558354297],
    [-12.90671039033975, 32.97377013244108],
    [2.071961008999537, 35.53114216168298],
    [13.977430855769256, 35.78497347916536],
    [25.742716153284814, 34.520995787410804],
    [33.72674056884805, 32.7

In [35]:
def timestamp_ms_to_yydoy(ts_ms):
    dt = datetime.utcfromtimestamp(ts_ms / 1000)
    doy = dt.timetuple().tm_yday
    return (dt.year - 2000) * 1000 + doy


def yydoy_to_datestr(yydoy):
    year = 2000 + yydoy // 1000
    doy = yydoy % 1000
    return (datetime(year, 1, 1) + timedelta(days=doy - 1)).strftime("%Y-%m-%d")


props = latest_radd_alert.getInfo()["properties"]
date_min = timestamp_ms_to_yydoy(props["system:time_start"])
date_max = timestamp_ms_to_yydoy(props["system:time_end"])

In [37]:
Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")
Map.add_basemap("Stadia.AlidadeSmoothDark")
Map.centerObject(roi)

# Date layer
Map.addLayer(
    latest_radd_alert.select("Date"),
    {"min": 24001, "max": 26365, "palette": ["lightyellow", "darkred"]},
    "Alert date",
)

Map.setCenter(10, 50, zoom=5)  # centered on Europe

alert_mask = latest_radd_alert.select("Alert").gte(2)
masked = latest_radd_alert.updateMask(alert_mask)

Map.addLayer(
    masked.select("Alert"),
    {"min": 2, "max": 3, "palette": ["cyan", "orange"]},
    "Alerts only",
)

roi_fc = ee.FeatureCollection(roi)
empty_img = ee.Image().byte()
roi_contour = empty_img.paint(**{"featureCollection": roi_fc, "color": 1, "width": 3})
Map.addLayer(roi_contour, {"palette": "blue"}, "Aquitaine")

# Alert legend (discrete)
Map.add_legend(
    title="Alert Confidence",
    legend_dict={
        "Unconfirmed (low)": "#00FFFF",  # cyan hex
        "Confirmed (high)": "#FFA500",  # orange hex
    },
)

# Date colorbar with actual date labels
Map.add_colorbar(
    vis_params={
        "min": date_min,
        "max": date_max,
        "palette": ["lightyellow", "darkred"],
    },
    label=f"Alert Date ({yydoy_to_datestr(date_min)} → {yydoy_to_datestr(date_max)})",
    orientation="horizontal",
)

Map

Map(center=[50, 10], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chil…